In [2]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer

In [3]:
df_train = pd.read_csv('data/train.csv')
df_test = pd.read_csv("data/test.csv")

df_test["Transported"] = False
df = pd.concat([df_train, df_test])
df.sample(5)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
2367,2549_02,Mars,False,F/522/P,TRAPPIST-1e,0.0,False,0.0,0.0,0.0,0.0,0.0,Rios Prin,True
2476,2657_01,Earth,False,F/514/S,TRAPPIST-1e,20.0,False,0.0,27.0,1.0,686.0,1.0,Ricard Gainney,False
3310,7248_01,Mars,False,F/1500/P,55 Cancri e,32.0,False,976.0,0.0,84.0,22.0,0.0,Hony Mité,False
4942,5268_01,Earth,False,F/1013/S,TRAPPIST-1e,14.0,False,0.0,56.0,NaN,0.0,2.0,Bony Flynney,True
3517,7636_01,Earth,False,E/498/P,TRAPPIST-1e,13.0,False,1118.0,0.0,93.0,5.0,393.0,Reney Davens,False


In [4]:
df.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name', 'Transported'],
      dtype='str')

In [5]:
df.info()

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  str    
 1   HomePlanet    12682 non-null  str    
 2   CryoSleep     12660 non-null  object 
 3   Cabin         12671 non-null  str    
 4   Destination   12696 non-null  str    
 5   Age           12700 non-null  float64
 6   VIP           12674 non-null  object 
 7   RoomService   12707 non-null  float64
 8   FoodCourt     12681 non-null  float64
 9   ShoppingMall  12664 non-null  float64
 10  Spa           12686 non-null  float64
 11  VRDeck        12702 non-null  float64
 12  Name          12676 non-null  str    
 13  Transported   12970 non-null  bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 1.4+ MB


In [6]:
df.isnull().sum()

PassengerId       0
HomePlanet      288
CryoSleep       310
Cabin           299
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Name            294
Transported       0
dtype: int64

In [7]:
df.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,12700.000000,12707.000000,12681.000000,12664.000000,12686.000000,12702.000000
mean,28.771969,222.897852,451.961675,174.906033,308.476904,306.789482
std,14.387261,647.596664,1584.370747,590.558690,1130.279641,1180.097223
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,49.000000,77.000000,29.000000,57.000000,42.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [8]:
df.sample(5)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
2831,6258_01,Earth,False,F/1291/P,PSO J318.5-22,19.0,False,0.0,599.0,0.0,0.0,0.0,Joeula Jacostaffey,False
8063,8618_02,Europa,True,B/336/S,TRAPPIST-1e,16.0,False,0.0,0.0,0.0,0.0,0.0,Jihonon Resendent,True
2875,6332_02,Earth,False,F/1310/P,TRAPPIST-1e,18.0,False,0.0,0.0,866.0,0.0,0.0,Andary Casez,False
5202,5546_01,Europa,False,D/176/P,55 Cancri e,28.0,False,550.0,5203.0,0.0,2244.0,1624.0,Wasatan Brathful,False
2675,2866_01,Europa,True,C/110/S,TRAPPIST-1e,36.0,True,0.0,0.0,0.0,0.0,0.0,Hadirk Wheededly,True


In [9]:
df[['Deck', 'Num', 'Side']] = df['Cabin'].str.split("/", expand=True)

In [10]:
df.drop(columns="Cabin", inplace=True)
df.sample(5)

,PassengerId,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck,Num,Side
1807,3858_02,Europa,True,TRAPPIST-1e,16.0,False,0.0,0.0,0.0,0.0,0.0,Furudah Sciatberit,False,E,231,P
8378,8956_08,Earth,False,TRAPPIST-1e,26.0,False,0.0,0.0,0.0,0.0,741.0,Harvin Bonnondry,False,F,1838,P
6299,6665_04,Europa,False,55 Cancri e,35.0,False,0.0,2504.0,0.0,0.0,5496.0,Saiphda Disivering,False,B,222,P
1580,3388_01,Europa,False,55 Cancri e,60.0,False,0.0,4230.0,0.0,NaN,NaN,Luxons Assefle,False,D,110,S
6229,6590_01,Europa,True,TRAPPIST-1e,24.0,False,0.0,0.0,0.0,0.0,0.0,Gollux Nifficaus,False,B,217,P


In [11]:
df.info()

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  str    
 1   HomePlanet    12682 non-null  str    
 2   CryoSleep     12660 non-null  object 
 3   Destination   12696 non-null  str    
 4   Age           12700 non-null  float64
 5   VIP           12674 non-null  object 
 6   RoomService   12707 non-null  float64
 7   FoodCourt     12681 non-null  float64
 8   ShoppingMall  12664 non-null  float64
 9   Spa           12686 non-null  float64
 10  VRDeck        12702 non-null  float64
 11  Name          12676 non-null  str    
 12  Transported   12970 non-null  bool   
 13  Deck          12671 non-null  str    
 14  Num           12671 non-null  str    
 15  Side          12671 non-null  str    
dtypes: bool(1), float64(6), object(2), str(7)
memory usage: 1.6+ MB


In [12]:
imputer = KNNImputer(n_neighbors=5, weights="distance")
num_cols = df[["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa","VRDeck"]]
filled_data = imputer.fit_transform(num_cols)

In [13]:
filled_data = pd.DataFrame(filled_data, columns=num_cols.columns, index=num_cols.index)
filled_data

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,39.000000,0.0,0.0,0.0,0.0,0.0
1,24.000000,109.0,9.0,25.0,549.0,44.0
2,58.000000,43.0,3576.0,0.0,6715.0,49.0
3,33.000000,0.0,1283.0,371.0,3329.0,193.0
4,16.000000,303.0,70.0,151.0,565.0,2.0
...,...,...,...,...,...,...
4272,34.000000,0.0,0.0,0.0,0.0,0.0
4273,42.000000,0.0,847.0,17.0,10.0,144.0
4274,35.200000,0.0,0.0,0.0,0.0,0.0
4275,23.666667,0.0,2680.0,0.0,0.0,523.0


In [14]:
df[num_cols.columns] = filled_data

In [15]:
df.isna().sum()

PassengerId       0
HomePlanet      288
CryoSleep       310
Destination     274
Age               0
VIP             296
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name            294
Transported       0
Deck            299
Num             299
Side            299
dtype: int64

In [16]:
df.sample(10)

,PassengerId,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck,Num,Side
5573,5934_01,Earth,False,TRAPPIST-1e,29.0,False,89.0,1725.0,38.614947,0.0,3.0,Joanry Mcfaddenson,True,G,962,P
5491,5858_01,Earth,False,55 Cancri e,22.0,False,483.0,0.0,0.000000,304.0,21.0,Wally Barberts,True,G,948,S
6269,6635_02,Europa,False,55 Cancri e,45.0,False,0.0,313.0,1791.000000,414.0,1675.0,Nusakab Entratable,False,C,209,P
5648,6006_01,Europa,False,55 Cancri e,45.0,True,1017.0,3863.0,0.000000,601.0,3999.0,Okulam Monpreery,False,D,190,P
3875,8461_01,Europa,False,55 Cancri e,39.0,False,0.0,11020.0,417.000000,182.0,4281.0,Meratz Cherticale,False,D,251,S
1120,2359_01,Earth,False,TRAPPIST-1e,22.0,False,1448.0,0.0,0.000000,0.0,1.0,Allene Macdonadows,False,F,480,P
7949,8489_01,Earth,True,PSO J318.5-22,22.0,False,0.0,0.0,0.000000,0.0,0.0,Elicey Danielps,True,G,1383,P
7195,7689_01,Earth,False,NaN,24.0,False,0.0,96.0,0.000000,597.0,0.0,Erie Sextones,False,F,1597,P
3755,4016_01,Earth,False,55 Cancri e,34.0,False,309.0,542.0,0.000000,0.0,55.0,Dorene Mclainez,False,E,268,S
1354,2914_01,Europa,True,TRAPPIST-1e,19.0,False,0.0,0.0,0.000000,0.0,0.0,Sadares Reeddommy,False,B,91,P


In [17]:
df["Name"] = df["Name"].fillna('U')

In [18]:
df.info()

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  str    
 1   HomePlanet    12682 non-null  str    
 2   CryoSleep     12660 non-null  object 
 3   Destination   12696 non-null  str    
 4   Age           12970 non-null  float64
 5   VIP           12674 non-null  object 
 6   RoomService   12970 non-null  float64
 7   FoodCourt     12970 non-null  float64
 8   ShoppingMall  12970 non-null  float64
 9   Spa           12970 non-null  float64
 10  VRDeck        12970 non-null  float64
 11  Name          12970 non-null  str    
 12  Transported   12970 non-null  bool   
 13  Deck          12671 non-null  str    
 14  Num           12671 non-null  str    
 15  Side          12671 non-null  str    
dtypes: bool(1), float64(6), object(2), str(7)
memory usage: 1.6+ MB


In [19]:
df['CryoSleep'].value_counts()

CryoSleep
False    8079
True     4581
Name: count, dtype: int64

In [20]:
df['Destination'].value_counts()

Destination
TRAPPIST-1e      8871
55 Cancri e      2641
PSO J318.5-22    1184
Name: count, dtype: int64

In [21]:
df['HomePlanet'].value_counts()

HomePlanet
Earth     6865
Europa    3133
Mars      2684
Name: count, dtype: int64

In [22]:
df["VIP"].value_counts()

VIP
False    12401
True       273
Name: count, dtype: int64

In [23]:
df["VIP"] = df["VIP"].fillna(False)

In [24]:
df.isna().sum()

PassengerId       0
HomePlanet      288
CryoSleep       310
Destination     274
Age               0
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name              0
Transported       0
Deck            299
Num             299
Side            299
dtype: int64

In [25]:
df['CryoSleep'].value_counts()

CryoSleep
False    8079
True     4581
Name: count, dtype: int64

In [26]:
df.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP',
       'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name',
       'Transported', 'Deck', 'Num', 'Side'],
      dtype='str')

In [27]:
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
df["CryoSleep"] = df["CryoSleep"].fillna(
    (df[spend_cols].sum(axis=1) == 0)
)

In [28]:
df['Deck'].value_counts(), df['Num'].value_counts(), df['Side'].value_counts()

(Deck
 F    4239
 G    3781
 E    1323
 B    1141
 C    1102
 D     720
 A     354
 T      11
 Name: count, dtype: int64,
 Num
 82      34
 4       28
 56      28
 31      27
 95      27
         ..
 1882     1
 1883     1
 1885     1
 1887     1
 1890     1
 Name: count, Length: 1894, dtype: int64,
 Side
 S    6381
 P    6290
 Name: count, dtype: int64)

In [29]:
df.info()

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   12970 non-null  str    
 1   HomePlanet    12682 non-null  str    
 2   CryoSleep     12970 non-null  object 
 3   Destination   12696 non-null  str    
 4   Age           12970 non-null  float64
 5   VIP           12970 non-null  object 
 6   RoomService   12970 non-null  float64
 7   FoodCourt     12970 non-null  float64
 8   ShoppingMall  12970 non-null  float64
 9   Spa           12970 non-null  float64
 10  VRDeck        12970 non-null  float64
 11  Name          12970 non-null  str    
 12  Transported   12970 non-null  bool   
 13  Deck          12671 non-null  str    
 14  Num           12671 non-null  str    
 15  Side          12671 non-null  str    
dtypes: bool(1), float64(6), object(2), str(7)
memory usage: 1.6+ MB


In [30]:
df['Group'] = df["PassengerId"].str.split("_").str[0]

In [31]:
df["Group"].value_counts()

Group
0984    8
4005    8
4256    8
4498    8
5133    8
       ..
9265    1
9269    1
9271    1
9273    1
9277    1
Name: count, Length: 9280, dtype: int64

In [32]:
df.isna().sum()

PassengerId       0
HomePlanet      288
CryoSleep         0
Destination     274
Age               0
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name              0
Transported       0
Deck            299
Num             299
Side            299
Group             0
dtype: int64

In [33]:
df['Deck'] = df.groupby("Group")['Deck'].transform(
    lambda x : x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan)
)

In [34]:
df['Deck'] = df["Deck"].fillna(df["Deck"].mode()[0])
df["Side"] = df["Side"].fillna(df["Side"].mode()[0])

In [35]:
df.drop(columns='Num', inplace=True)

In [36]:
df['HomePlanet'] = df.groupby(["Group", "Deck", "Side"])["HomePlanet"].transform(
    lambda x:x.fillna(
        x.mode().iloc[0] if not x.mode().empty else np.nan
        ))

In [37]:
df["HomePlanet"] = df["HomePlanet"].fillna(df["HomePlanet"].mode()[0])

In [38]:
df["Destination"] = df.groupby(["Group", "HomePlanet"])["Destination"].transform(
    lambda x : x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan)
)

In [39]:
df["Destination"] = df["Destination"].fillna(df["Destination"].mode()[0])

In [40]:
df.isna().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
Deck            0
Side            0
Group           0
dtype: int64

In [41]:
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
df["Spendings"] = df[spend_cols].sum(axis=1)
df.drop(columns=spend_cols, inplace=True)

In [42]:
df.info()

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  12970 non-null  str    
 1   HomePlanet   12970 non-null  str    
 2   CryoSleep    12970 non-null  object 
 3   Destination  12970 non-null  str    
 4   Age          12970 non-null  float64
 5   VIP          12970 non-null  object 
 6   Name         12970 non-null  str    
 7   Transported  12970 non-null  bool   
 8   Deck         12970 non-null  str    
 9   Side         12970 non-null  str    
 10  Group        12970 non-null  object 
 11  Spendings    12970 non-null  float64
dtypes: bool(1), float64(2), object(3), str(6)
memory usage: 1.2+ MB


In [43]:
df["CryoSleep"] = df["CryoSleep"].astype(bool)
df["CryoSleep"] = df["CryoSleep"].astype(int)

df["VIP"] = df["VIP"].astype(bool)
df["VIP"] = df["VIP"].astype(int)

df["Transported"] = df["Transported"].astype(bool)
df["Transported"] = df["Transported"].astype(int)

In [44]:
df["Side"].value_counts()

Side
S    6680
P    6290
Name: count, dtype: int64

In [45]:
df["Side"] = df["Side"].map({
    "P" :0,
    "S" :1
})

In [46]:
df = pd.get_dummies(df, columns=["HomePlanet", "Destination", "Deck"], dtype=int)


In [47]:
df.columns

Index(['PassengerId', 'CryoSleep', 'Age', 'VIP', 'Name', 'Transported', 'Side',
       'Group', 'Spendings', 'HomePlanet_Earth', 'HomePlanet_Europa',
       'HomePlanet_Mars', 'Destination_55 Cancri e',
       'Destination_PSO J318.5-22', 'Destination_TRAPPIST-1e', 'Deck_A',
       'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T'],
      dtype='str')

In [48]:
# bool_cols = df.select_dtypes(bool).columns
# df[bool_cols] = df[bool_cols].astype(int)

In [49]:
df["PassengerId"].sample(5)

683     1392_01
1748    1861_01
5728    6068_01
1144    1211_04
3151    6888_04
Name: PassengerId, dtype: str

In [50]:
df["Group"] = df["Group"].astype(int)
df["GroupSize"] = df.groupby("Group")["Group"].transform("count")
df["GroupMember"] = df["PassengerId"].str.split("_").str[1]

In [58]:
df["GroupMember"] = df["GroupMember"].astype(int)

In [59]:
cols_to_drop = ['Name', 'PassengerId']
df = df.drop(columns=cols_to_drop)

KeyError: "['Name', 'PassengerId'] not found in axis"

In [ ]:
df.columns

Index(['CryoSleep', 'Age', 'VIP', 'Transported', 'Side', 'Group', 'Spendings',
       'HomePlanet_Earth', 'HomePlanet_Europa', 'HomePlanet_Mars',
       'Destination_55 Cancri e', 'Destination_PSO J318.5-22',
       'Destination_TRAPPIST-1e', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D',
       'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'GroupSize', 'GroupMember'],
      dtype='str')

In [60]:
df.info()

<class 'pandas.DataFrame'>
Index: 12970 entries, 0 to 4276
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CryoSleep                  12970 non-null  int64  
 1   Age                        12970 non-null  float64
 2   VIP                        12970 non-null  int64  
 3   Transported                12970 non-null  int64  
 4   Side                       12970 non-null  int64  
 5   Group                      12970 non-null  int64  
 6   Spendings                  12970 non-null  float64
 7   HomePlanet_Earth           12970 non-null  int64  
 8   HomePlanet_Europa          12970 non-null  int64  
 9   HomePlanet_Mars            12970 non-null  int64  
 10  Destination_55 Cancri e    12970 non-null  int64  
 11  Destination_PSO J318.5-22  12970 non-null  int64  
 12  Destination_TRAPPIST-1e    12970 non-null  int64  
 13  Deck_A                     12970 non-null  int64  
 14  Deck_B 

In [61]:
df.to_csv("data/cleaned_combined.csv", index=False)